# ⚗️ C3 Inclusion-Site Compatibility Screening

This tool evaluates the geometric and chemical compatibility of small molecules with the guest-binding site of the
PET cyclic trimer (C3) inclusion phase. Compatibility is assessed based on the structural criteria used in this
screening model.

**How to use**

1. In the menu, choose **Runtime → Run all**. The first run takes about a minute (it installs RDKit).
   If Colab warns that the notebook was not authored by Google, click **Run anyway**.
2. A text box appears below the code cell. Type a SMILES and click **Check**.
   Several molecules at once: separate the SMILES with spaces or commas.

**Classification categories**

* ✅ **COMPATIBLE** – both termini occupy their pockets within the reference limits.
* ❌ **NOT COMPATIBLE** – one or more essential geometric or chemical criteria are not satisfied.
* ➖ **BORDERLINE** – modest deviations from the reference geometric limits.
* ❔ **OUTSIDE THE MODEL** – salts, mixtures, charged species, elements other than H, C, N, O, F, S, Cl, Br, I.


In [ ]:
#@title ▶ Setup and screening tool (code hidden — just run it) { display-mode: "form" }
import subprocess, sys
try:
    import rdkit
except ImportError:
    print("Installing RDKit (about a minute, only the first time)…", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=True)
import importlib; importlib.invalidate_caches()

# ---------------------------------------------------------------------------
# C3 inclusion-site compatibility screening: geometric and chemical compatibility
# of small molecules with the guest site of the PET cyclic trimer (C3) inclusion phase.
#
# Model (host geometry + one chemical criterion):
#   * In the inclusion phase each end of the guest sits in a pocket inside one
#     C3 ring, lined by three aromatic C-H groups. The two pockets belong to two
#     neighbouring C3 rings; between them is a flat slot.
#   * The host is taken from the DFT-relaxed trans-DCE@C3 structure and kept
#     frozen. The guest is built from SMILES (RDKit, MMFF conformers within
#     3 kcal/mol), its two most distant heavy atoms are placed into the two
#     pockets, and it is rotated/shifted to the best fit.
#   * Descriptors: pocket contact of each terminus, terminal compression, and
#     overlap of the rest of the molecule with the host framework.
#   * Reference limits: largest values among the experimentally characterized
#     guests (1,2-dichloroethane, trans-1,2-dichloroethylene, 1,2-dibromoethane,
#     1-bromo-2-chloroethane): terminal compression 0.25 A, framework overlap 0.19 A.
#   * Chemical criterion: terminal atoms must carry lone pairs (halogen, O, N, S).
# ---------------------------------------------------------------------------
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

VDW = {'H': 1.20, 'C': 1.70, 'N': 1.55, 'O': 1.52, 'F': 1.47, 'S': 1.80,
       'Cl': 1.75, 'Br': 1.85, 'I': 1.98}                    # Bondi radii, A
ALLOWED = set(VDW)
LONE_PAIR_ENDS = {'F', 'Cl', 'Br', 'I', 'O', 'N', 'S'}
NAMES = {'F': 'fluorine', 'Cl': 'chlorine', 'Br': 'bromine', 'I': 'iodine',
         'O': 'oxygen', 'N': 'nitrogen', 'S': 'sulfur', 'C': 'carbon'}

# limits (A), set by the guests confirmed in this host
POCKET_TOUCH = 0.05     # end must be within this of contact with all three pocket C-H
SQUEEZE_CL, SQUEEZE_BR = 0.12, 0.25        # largest values among the confirmed guests
SQUEEZE_BORDER, SQUEEZE_MAX = 0.255, 0.30
WALL_OK, WALL_MAX = 0.19, 0.35
MAX_HEAVY = 10

_host_el, _host_xyz, _host_r, _pockets = None, None, None, None


def load_host(text):
    """text: lines 'El x y z' in the site frame (pockets on the z axis at z = +/-2.1406)."""
    global _host_el, _host_xyz, _host_r, _pockets
    el, xyz = [], []
    for line in text.strip().splitlines():
        s, x, y, z = line.split()
        el.append(s); xyz.append((float(x), float(y), float(z)))
    _host_el, _host_xyz = np.array(el), np.array(xyz)
    _host_r = np.array([VDW[s] for s in el])
    H = np.where(_host_el == 'H')[0]
    _pockets = []
    for z in (-2.1406, 2.1406):
        d = np.linalg.norm(_host_xyz[H] - np.array([0, 0, z]), axis=1)
        _pockets.append(_host_xyz[H[np.argsort(d)[:3]]])


def _rot_z_to(v):
    v = v / np.linalg.norm(v); z = np.array([0., 0., 1.])
    c = float(np.dot(v, z)); w = np.cross(v, z)
    if np.linalg.norm(w) < 1e-9:
        return np.eye(3) if c > 0 else np.diag([1., -1., -1.])
    K = np.array([[0, -w[2], w[1]], [w[2], 0, -w[0]], [-w[1], w[0], 0]])
    return np.eye(3) + K + K @ K / (1 + c)


def _rz(phi):
    c, s = np.cos(phi), np.sin(phi)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def _rx(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def _ry(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def _screen(smi):
    """Return (mol, problem_message_or_None, verdict_if_rejected)."""
    smi = smi.strip()
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None, "Could not read this SMILES. Please check the spelling.", "unreadable"
    if len(Chem.GetMolFrags(mol)) > 1:
        return None, "More than one molecule (salt or mixture). The model handles one neutral molecule.", "outside"
    if any(a.GetFormalCharge() != 0 for a in mol.GetAtoms()):
        return None, "Charged species are outside the model (it covers neutral molecules).", "outside"
    bad = sorted({a.GetSymbol() for a in mol.GetAtoms()} - ALLOWED)
    if bad:
        return None, f"Contains {', '.join(bad)}; the model covers H, C, N, O, F, S, Cl, Br, I.", "outside"
    nheavy = mol.GetNumHeavyAtoms()
    if nheavy < 2:
        return None, "A single heavy atom cannot bridge the two pockets.", "small"
    if nheavy > MAX_HEAVY:
        return None, (f"{nheavy} heavy atoms. The cavity holds a rod of about four heavy atoms "
                      f"(X-C-C-X), so this molecule is far too large."), "large"
    return mol, None, None


def analyse(smi, nconf=60, ewin=3.0, seed=7):
    mol, problem, kind = _screen(smi)
    if problem:
        return dict(smiles=smi, status=kind, message=problem)
    m = Chem.AddHs(mol)
    cids = list(AllChem.EmbedMultipleConfs(m, nconf, randomSeed=seed))
    if not cids:
        return dict(smiles=smi, status='outside', message="Could not build a 3D structure for this molecule.")
    res = AllChem.MMFFOptimizeMoleculeConfs(m, maxIters=2000)
    E = np.array([e for _, e in res])
    S = [a.GetSymbol() for a in m.GetAtoms()]
    R = np.array([VDW[s] for s in S])
    heavy = [i for i, s in enumerate(S) if s != 'H']

    # candidate placements: each accessible conformer, both end-to-pocket assignments
    cands = []
    for cid, e in zip(cids, E):
        if e - E.min() > ewin:
            continue
        X = m.GetConformer(cid).GetPositions()
        L, i, j = max((np.linalg.norm(X[a] - X[b]), a, b)
                      for k, a in enumerate(heavy) for b in heavy[k + 1:])
        for a, b in ((i, j), (j, i)):
            Y = X - (X[a] + X[b]) / 2
            cands.append((Y @ _rot_z_to(Y[b] - Y[a]).T, a, b, L, e - E.min()))
    Lmax = max(c[3] for c in cands)
    near = np.linalg.norm(_host_xyz, axis=1) < Lmax / 2 + 7.0
    HP, HR = _host_xyz[near], _host_r[near]
    side_of = lambda a, b: np.array([k for k in range(len(S)) if k not in (a, b)], dtype=int)

    H2 = (HP ** 2).sum(1)

    def score(poses, a, b, chunk=150):
        """poses: (P, n, 3) -> worst overlap per pose, plus per-atom max overlaps.
        Distances via |p|^2 + |h|^2 - 2 p.h in chunks, to keep memory low."""
        P, n, _ = poses.shape
        ov = np.empty((P, n))
        for s in range(0, P, chunk):
            Q = poses[s:s + chunk].reshape(-1, 3)
            d2 = (Q ** 2).sum(1)[:, None] + H2[None, :] - 2.0 * Q @ HP.T
            D = np.sqrt(np.maximum(d2, 0.0)).reshape(-1, n, len(HP))
            ov[s:s + chunk] = (R[None, :, None] + HR[None, None, :] - D).max(2)
        sd = side_of(a, b)
        wall = ov[:, sd].max(1) if len(sd) else np.full(len(poses), -9.0)
        return np.maximum(wall, np.maximum(ov[:, a], ov[:, b])), ov, wall

    def poses_for(Y, phis, shifts, tilts):
        out, par = [], []
        for tx, ty in tilts:
            T = _rx(tx) @ _ry(ty)
            for phi in phis:
                Z = Y @ (T @ _rz(phi)).T
                for sh in shifts:
                    out.append(Z + np.array([0, 0, sh])); par.append((phi, sh, tx, ty))
        return np.array(out), par

    # 1) coarse search
    coarse = []
    for ci, (Y, a, b, L, dE) in enumerate(cands):
        P, par = poses_for(Y, np.radians(np.arange(0, 360, 10)), np.arange(-0.3, 0.31, 0.1), [(0, 0)])
        sc, _, _ = score(P, a, b)
        for k in np.argsort(sc)[:3]:
            coarse.append((sc[k], ci, par[k]))
    coarse.sort(key=lambda t: t[0])

    # 2) refine the best few placements (finer rotation, shift, small tilt of the axis)
    tilts = [(np.radians(x), np.radians(y)) for x in (-6, -3, 0, 3, 6) for y in (-6, -3, 0, 3, 6)]
    best = None
    for s0, ci, (phi0, sh0, _, _) in coarse[:5]:
        Y, a, b, L, dE = cands[ci]
        P, par = poses_for(Y, phi0 + np.radians(np.arange(-6, 6.1, 1.5)),
                           sh0 + np.arange(-0.1, 0.101, 0.025), tilts)
        sc, ov, wall = score(P, a, b)
        k = int(np.argmin(sc))
        if best is None or sc[k] < best['score']:
            W = P[k]
            gapA = (np.linalg.norm(_pockets[0] - W[a], axis=1) - R[a] - 1.20).max()
            gapB = (np.linalg.norm(_pockets[1] - W[b], axis=1) - R[b] - 1.20).max()
            best = dict(score=float(sc[k]), L=L, ends=(S[a], S[b]), squeeze=(ov[k, a], ov[k, b]),
                        gap=(gapA, gapB), wall=float(wall[k]), dE=dE)
    best.update(smiles=smi, status='done')
    return best


def judge(r, carbon_end_rule=True):
    """Adds r['verdict'] in {'fits','no','border', ...}, r['checks'] = [(mark, name, text)],
    r['reasons'] (short) and r['why'] (explanations). carbon_end_rule=False is kept only for testing."""
    if r['status'] != 'done':
        r['verdict'] = {'large': 'no', 'small': 'no'}.get(r['status'], r['status'])
        r['reasons'] = {'large': ['too large'], 'small': ['too small']}.get(r['status'], [])
        r['checks'], r['why'] = [], [r['message']]
        return r
    e1, e2 = r['ends']; s1, s2 = r['squeeze']; g1, g2 = r['gap']; wall = r['wall']
    fails, borders, chk, why = [], [], [], []

    # 1. chemical criterion: terminal atoms must accept C-H...X contacts
    carbon = [e for e in (e1, e2) if e not in LONE_PAIR_ENDS]
    if carbon and carbon_end_rule:
        chk.append(('✗', 'Terminal atoms', f"{e1} and {e2}: a carbon terminus cannot accept C–H···X contacts"))
        fails.append('carbon terminus')
        why.append("A terminal atom is carbon. Each pocket is formed by three aromatic C–H groups directed at the "
                   "terminal atom; a carbon terminus has no lone pairs to accept C–H···X contacts. In DFT, ethane and "
                   "ethylene bind at −10.0 and −8.3 kcal/mol, compared with −20.8 and −18.2 kcal/mol for "
                   "1,2-dichloroethane and trans-1,2-dichloroethylene.")
    else:
        chk.append(('✓', 'Terminal atoms', f"{e1} and {e2}: can accept C–H···X contacts"))

    # 2. pocket contact
    far = [(e, g) for e, g in ((e1, g1), (e2, g2)) if g > POCKET_TOUCH]
    if far:
        if len(far) == 2 and far[0][0] == far[1][0]:
            txt = f"both {far[0][0]} termini: gap {far[0][1]:.2f} / {far[1][1]:.2f} Å (contact required)"
        else:
            txt = ", ".join(f"{e} terminus: gap {g:.2f} Å" for e, g in far) + " (contact required)"
        chk.append(('✗', 'Pocket contact', txt))
        fails.append('terminus does not reach its pocket')
        why.append("At least one terminus cannot reach its pocket while the other occupies its own: the molecule "
                   "is too short, or its terminal atom too small.")
    else:
        chk.append(('✓', 'Pocket contact', "both termini in contact with the C–H groups of their pockets"))

    # 3. terminal compression
    smax = max(s1, s2)
    ref = f"(reference: Cl {SQUEEZE_CL:.2f}, Br {SQUEEZE_BR:.2f} Å)"
    if smax > SQUEEZE_MAX:
        chk.append(('✗', 'Terminal compression',
                    f"{s1:.2f} / {s2:.2f} Å (limit {SQUEEZE_MAX:.2f} Å)"))
        fails.append('excessive terminal compression')
        why.append(f"Terminal compression of {smax:.2f} Å exceeds the geometric incompatibility limit "
                   f"of {SQUEEZE_MAX:.2f} Å. The guest is therefore incompatible with the "
                   f"fixed C3 binding-site geometry under the screening criteria.")
    elif smax > SQUEEZE_BORDER:
        chk.append(('~', 'Terminal compression', f"{s1:.2f} / {s2:.2f} Å {ref}"))
        borders.append('terminal compression near limit')
    else:
        chk.append(('✓', 'Terminal compression', f"{max(s1, 0):.2f} / {max(s2, 0):.2f} Å {ref}"))

    # 4. framework overlap
    lim = f"(reference ≤ {WALL_OK:.2f} Å)"
    if wall > WALL_MAX:
        chk.append(('✗', 'Framework overlap', f"{wall:.2f} Å {lim}"))
        fails.append('framework overlap')
        why.append("Part of the molecule overlaps with the host framework between the pockets — typically a side "
                   "group, a bent geometry, or a molecule that is too long or too bulky.")
    elif wall > WALL_OK:
        chk.append(('~', 'Framework overlap', f"{wall:.2f} Å {lim}"))
        borders.append('framework overlap near limit')
    else:
        chk.append(('✓', 'Framework overlap', f"{max(wall, 0):.2f} Å {lim}"))

    if fails:
        r['verdict'] = 'no'; r['reasons'] = fails
    elif borders:
        r['verdict'] = 'border'; r['reasons'] = borders
        why.append("The geometry is close to the reference limits and is classified as borderline.")
    else:
        r['verdict'] = 'fits'; r['reasons'] = []
        why.append("Both termini occupy the C–H pockets of two neighbouring C3 rings without significant overlap "
                   "with the host framework.")
    r['checks'], r['why'] = chk, why
    return r


ICON = {'fits': '✅', 'no': '❌', 'border': '➖', 'outside': '❔', 'unreadable': '❔'}
LABEL = {'fits': 'COMPATIBLE', 'no': 'NOT COMPATIBLE',
         'border': 'BORDERLINE', 'outside': 'OUTSIDE THE MODEL', 'unreadable': 'COULD NOT READ'}


def checks(r):
    return r.get('checks', [])


def report(r):
    bar = '═' * 76
    out = [bar, f"{r['smiles']}"]
    head = f"{ICON[r['verdict']]} {LABEL[r['verdict']]}"
    if r.get('reasons') and r['verdict'] != 'fits':
        head += ' — ' + '; '.join(r['reasons'])
    out.append(f"RESULT:  {head}")
    if r['status'] == 'done':
        out.append(f"\n  Termini: {r['ends'][0]} … {r['ends'][1]}, {r['L']:.2f} Å apart "
                   f"(most extended conformer within 3 kcal/mol)")
        out += [f"  {m} {name:<21s}{text}" for m, name, text in r['checks']]
    out.append('')
    for w in r['why']:
        out += _wrap(w, 74, '  ')
    return '\n'.join(out)


def _wrap(text, width, indent):
    words, lines, cur = text.split(), [], indent
    for w in words:
        if len(cur) + len(w) + 1 > width + len(indent):
            lines.append(cur.rstrip()); cur = indent
        cur += w + ' '
    lines.append(cur.rstrip())
    return lines


def split_input(text):
    return [s for s in text.replace(',', ' ').replace(';', ' ').split() if s]


def run(text):
    items = split_input(text)
    if not items:
        print("Type one or more SMILES in the box above (separated by spaces or commas).")
        return
    results = []
    for smi in items:
        r = judge(analyse(smi))
        results.append(r)
        print(report(r), flush=True)
    if len(results) > 1:
        print('═' * 76 + '\nSUMMARY')
        for r in results:
            extra = f"  ({'; '.join(r['reasons'])})" if r.get('reasons') and r['verdict'] != 'fits' else ''
            print(f"  {ICON[r['verdict']]} {LABEL[r['verdict']]:<25s} {r['smiles']}{extra}")


# host atoms around the binding site (from the DFT-relaxed trans-DCE@C3 structure),
# frame: pockets on the z axis at z = -2.1406 and +2.1406 A
HOST_SITE = """
O 1.059 -9.129 3.044
C 1.669 -11.669 4.236
C 1.468 -11.496 2.859
C 1.141 -10.137 2.348
H 1.557 -10.810 4.896
O 0.614 -5.067 10.557
O -1.949 -4.912 9.401
O -5.511 -9.754 5.272
C 0.806 -5.053 11.899
C -1.341 -3.738 9.988
C -2.194 -4.886 8.067
C -2.832 -6.155 7.621
C -4.186 -7.317 5.994
C -2.833 -7.299 8.436
C -3.525 -8.437 8.043
C -4.218 -8.446 6.824
C -5.026 -9.618 6.393
H 1.114 -7.383 10.470
H -1.703 -3.727 11.022
H -2.299 -7.283 9.384
O 1.986 -9.269 8.466
O -3.047 -11.119 3.504
O -2.836 -8.858 3.296
C 0.122 -10.115 7.200
C -0.113 -8.895 6.545
C -1.013 -8.839 5.489
C -1.512 -11.194 5.778
C -1.700 -9.992 5.081
C -2.584 -9.903 3.887
H 0.425 -7.998 6.847
H -1.177 -7.900 4.968
H -5.915 -6.418 9.428
C -2.019 -0.768 12.493
H -2.469 -1.637 12.000
C -1.683 -5.853 2.942
H -2.044 -5.788 3.975
H -2.134 -6.721 2.450
O -5.407 -5.572 1.765
O -5.234 -4.838 3.917
O -6.904 1.590 1.333
C -6.379 -7.660 2.582
C -5.118 -6.929 2.189
C -5.459 -4.617 2.733
C -5.822 -3.289 2.167
C -5.956 -2.197 3.036
C -6.294 -0.949 2.526
C -6.496 -0.777 1.148
C -6.822 0.582 0.638
H -6.143 -8.674 2.927
H -6.917 -7.135 3.382
H -4.423 -6.921 3.035
H -5.784 -2.341 4.102
H -6.406 -0.091 3.185
O -5.742 -0.487 11.315
O -7.350 5.653 8.847
O -1.956 -3.926 7.343
C -10.059 -2.487 7.666
C -9.147 -2.551 8.713
C -6.369 1.967 10.341
C -6.703 3.212 9.830
C -3.498 -6.176 6.388
H -10.241 -1.541 7.161
H -8.593 -1.661 8.998
H -4.974 -2.323 10.878
H -6.267 1.122 9.669
H -6.849 3.336 8.759
H -1.684 -2.828 9.480
H -3.524 -5.278 5.771
H -4.744 -7.327 5.062
O -6.729 -0.677 7.114
O -5.977 1.451 6.755
O -11.011 -0.400 1.793
O -10.799 1.861 1.586
O -5.480 -3.228 7.097
O -5.148 -3.992 9.218
C -6.759 0.537 6.510
C -7.841 0.605 5.489
C -8.076 1.824 4.835
C -8.977 1.880 3.779
C -8.564 -0.539 5.114
C -9.475 -0.475 4.067
C -9.663 0.727 3.370
C -10.547 0.817 2.176
C -7.828 -7.590 6.869
C -6.473 -6.428 8.496
C -7.826 -6.447 6.055
C -7.134 -5.308 6.447
C -6.441 -5.299 7.666
C -5.634 -4.127 8.097
C -4.736 -2.020 7.398
C -5.633 -0.973 8.016
H -7.538 2.721 5.137
H -9.140 2.819 3.258
H -8.381 -1.486 5.619
H -10.029 -1.365 3.782
H -8.360 -6.463 5.107
H -7.098 -4.436 5.799
H -3.904 -2.239 8.079
H -4.346 -1.685 6.429
H -5.056 -0.064 8.222
H -6.093 -1.333 8.943
C -9.646 4.866 1.232
H -10.008 4.932 2.264
H -10.097 3.998 0.740
O 0.949 -10.151 1.007
O -1.614 -9.997 -0.149
C 1.596 -12.592 1.990
C -1.005 -8.823 0.438
C -1.859 -9.971 -1.483
C -2.496 -11.240 -1.929
C -2.498 -12.383 -1.115
H 1.449 -12.468 0.920
H -1.367 -8.812 1.472
H -1.964 -12.367 -0.167
H -5.580 -11.503 -0.122
C -3.944 -11.170 2.358
H -4.181 -10.157 2.013
H -3.407 -11.695 1.558
C -1.347 -10.937 -6.608
H -1.709 -10.872 -5.576
O -7.276 -7.711 1.437
O -7.488 -9.972 1.644
O -7.014 0.568 -0.703
O -9.577 0.722 -1.859
O -1.620 -9.010 -2.207
C -9.723 -7.571 -1.884
C -8.812 -7.636 -0.838
C -8.624 -8.838 -0.141
C -7.740 -8.928 1.053
C -6.033 -3.118 0.791
C -6.367 -1.873 0.280
C -8.968 1.897 -1.273
C -9.822 0.748 -3.194
C -10.459 -0.521 -3.640
C -3.162 -11.260 -3.162
C -10.461 -1.664 -2.825
C -11.153 -2.803 -3.217
C -11.846 -2.812 -4.437
H -9.906 -6.625 -2.389
H -8.258 -6.745 -0.553
H -4.639 -7.408 1.328
H -5.932 -3.963 0.118
H -6.514 -1.749 -0.791
H -9.330 1.908 -0.239
H -1.348 -7.912 -0.070
H -3.188 -10.363 -3.779
H -9.927 -1.648 -1.877
H -11.189 -3.675 -2.569
O -6.393 -5.762 -2.436
O -5.641 -3.634 -2.795
O -5.145 -8.312 -2.454
O -4.812 -9.077 -0.332
C -6.423 -4.547 -3.040
C -7.505 -4.480 -4.061
C -7.740 -3.261 -4.716
C -8.641 -3.205 -5.771
C -8.228 -5.624 -4.436
C -9.139 -5.560 -5.483
C -9.327 -4.358 -6.180
C -6.798 -10.393 -3.103
C -6.105 -10.384 -1.884
C -5.298 -9.212 -1.453
C -4.401 -7.105 -2.152
C -5.298 -6.058 -1.534
H -7.202 -2.364 -4.414
H -8.804 -2.266 -6.293
H -8.046 -6.570 -3.932
H -9.694 -6.450 -5.768
H -6.762 -9.521 -3.752
H -3.568 -7.324 -1.471
H -4.011 -6.769 -3.121
H -4.720 -5.149 -1.329
H -5.757 -6.418 -0.607
O -4.899 -9.922 -5.633
O -6.568 -3.494 -8.217
C -5.124 -9.702 -6.817
C -5.486 -8.374 -7.383
C -5.620 -7.282 -6.514
C -5.959 -6.034 -7.024
C -6.160 -5.862 -8.402
C -6.486 -4.503 -8.912
H -5.449 -7.425 -5.448
H -6.071 -5.176 -6.365
C -11.907 -0.451 0.648
H -12.144 0.563 0.303
H -11.370 -0.976 -0.152
C -9.311 -0.218 -8.318
H -9.672 -0.153 -7.286
O -9.583 1.709 -3.917
C -11.126 -0.541 -4.872
H -9.312 2.807 -1.781
H -11.151 0.357 -5.489
H -11.532 3.395 -3.182
O 10.203 -5.073 1.429
C 10.285 -6.082 0.734
O 5.760 -1.979 11.439
O 5.972 0.282 11.232
O 3.891 -4.118 11.111
O 3.718 -4.852 8.959
C 4.864 -2.030 10.294
C 3.603 -2.760 10.686
C 3.943 -5.073 10.142
H 3.123 -2.282 11.548
H 4.627 -1.016 9.949
H 5.401 -2.555 9.493
H 2.907 -2.769 9.841
H 4.269 -7.349 8.774
O 5.213 -9.013 5.762
O 7.194 -0.857 7.787
O 3.964 -6.462 5.779
O 3.632 -5.698 3.657
C 7.803 0.318 8.373
C 6.949 -0.831 6.452
C 6.312 -2.100 6.006
C 4.958 -3.262 4.380
C 6.310 -3.243 6.821
C 5.618 -4.382 6.429
C 4.925 -4.391 5.209
C 4.118 -5.563 4.778
C 3.221 -7.670 5.477
C 4.118 -8.717 4.860
C 9.311 0.218 8.318
H 6.865 -8.204 7.257
H 7.441 0.328 9.407
H 6.844 -3.227 7.769
H 5.582 -5.254 7.077
H 2.388 -7.451 4.796
H 2.831 -8.005 6.446
H 3.540 -9.626 4.654
H 4.577 -8.357 3.932
H 9.672 0.153 7.286
O 6.096 -7.064 1.889
O 6.307 -4.803 1.681
O 0.440 -5.764 5.532
C 9.266 -6.059 5.585
C 9.031 -4.840 4.930
C 8.130 -4.784 3.875
C 8.543 -7.203 5.210
C 7.632 -7.139 4.163
C 7.444 -5.937 3.466
C 6.560 -5.847 2.272
C 4.853 -11.657 2.534
C 1.982 -3.514 6.487
H 9.569 -3.943 5.232
H 7.967 -3.845 3.353
H 7.078 -8.029 3.878
H 3.458 -7.367 1.997
H 4.751 -10.812 3.207
H 0.168 -6.862 3.396
H 2.008 -4.412 7.104
H 3.228 -2.363 7.813
C 0.167 -3.837 9.933
H 0.529 -3.902 8.901
H 0.618 -2.969 10.426
H 11.532 -3.395 3.182
O 9.583 -1.709 3.917
C 11.126 0.541 4.872
H 9.312 -2.807 1.781
H 11.151 -0.357 5.489
O 3.737 -1.516 0.150
O 3.909 -0.782 2.302
O 2.240 5.646 -0.282
C 2.764 -3.604 0.967
C 4.025 -2.874 0.574
C 3.684 -0.562 1.119
C 3.322 0.766 0.552
C 3.188 1.858 1.421
C 2.849 3.106 0.911
C 2.648 3.278 -0.466
C 2.322 4.637 -0.977
H 3.001 -4.618 1.312
H 2.227 -3.080 1.767
H 4.720 -2.866 1.420
H 3.359 1.715 2.487
H 2.737 3.964 1.571
O -2.465 0.461 11.869
O -4.072 6.601 9.400
O -4.245 5.867 7.248
O -2.575 -0.561 9.832
C -3.445 4.147 10.374
C -3.112 2.902 10.885
C -0.511 -0.867 12.438
C -3.099 8.689 8.583
C -4.361 7.959 8.976
C -4.020 5.647 8.432
C -3.657 4.319 8.998
C -3.523 3.227 8.129
C -3.185 1.979 8.639
C -2.983 1.806 10.017
C -2.657 0.447 10.527
H -3.547 4.992 11.047
H -2.965 2.778 11.956
H -0.149 -0.878 11.404
H -2.562 8.164 7.783
H -5.056 7.950 8.130
H -3.695 3.370 7.063
H -3.073 1.120 7.980
O 3.401 3.568 9.700
O -2.750 1.707 4.051
O -3.502 -0.421 4.410
O 1.532 1.429 9.372
O 1.320 -0.832 9.579
O 1.794 9.708 7.232
O -0.769 9.863 6.076
O 7.188 0.130 5.729
O -3.999 4.257 4.068
O -4.331 5.021 1.947
C 2.428 1.480 10.517
C 3.690 2.211 10.125
C 3.349 4.523 10.669
C 2.986 5.851 10.103
C 2.312 8.363 9.084
C -2.720 0.492 4.655
C -1.638 0.425 5.676
C -1.403 -0.795 6.331
C -0.502 -0.851 7.386
C -0.915 1.569 6.051
C -0.004 1.505 7.098
C 0.184 0.302 7.795
C 1.068 0.213 8.989
C 2.774 6.022 8.727
C 2.441 7.267 8.215
C -0.160 11.037 6.663
C -1.014 9.888 4.742
C -1.651 8.620 4.296
C 5.646 -2.120 4.773
C -3.006 7.457 2.669
C -1.653 7.476 5.110
C -2.345 6.337 4.718
C -3.038 6.329 3.499
C -3.845 5.156 3.068
C -4.743 3.049 3.767
C -3.846 2.002 3.149
C 1.347 10.937 6.608
H 2.665 0.467 10.862
H 1.891 2.005 11.318
H 4.385 2.219 10.970
H -1.941 -1.692 6.029
H -0.339 -1.790 7.907
H -1.098 2.515 5.546
H 0.550 2.395 7.383
H 4.169 1.732 9.263
H 2.876 5.177 8.054
H 2.294 7.392 7.145
H 7.459 1.228 7.865
H 5.620 -1.222 4.157
H 4.400 -3.272 3.448
H -1.119 7.492 6.058
H -2.381 5.465 5.366
H -5.575 3.268 3.086
H -5.132 2.714 4.736
H -4.423 1.093 2.943
H -3.386 2.362 2.222
H 1.709 10.872 5.576
O 2.415 3.378 5.499
O 3.166 5.506 5.140
O -1.867 3.656 0.178
O -1.656 5.917 -0.029
O -2.129 -4.623 2.318
O 0.433 -4.778 3.474
O -7.523 4.955 3.822
O 3.663 0.828 5.482
O 3.996 0.064 7.603
C 2.384 4.593 4.896
C 1.303 4.660 3.875
C 1.067 5.879 3.220
C 0.167 5.935 2.164
C 0.580 3.516 3.499
C -0.332 3.580 2.452
C -0.520 4.783 1.755
C -1.404 4.872 0.561
C -3.110 -0.937 0.824
C -2.776 -2.183 1.335
C -0.175 -5.952 2.887
C 0.679 -4.804 4.808
C 1.316 -3.535 5.254
C -5.981 7.205 4.777
C 2.670 -2.373 6.881
C 1.317 -2.391 4.440
C 2.009 -1.253 4.832
C 2.703 -1.244 6.052
C 3.510 -0.072 6.483
C 4.407 2.035 5.784
C 3.510 3.082 6.401
H 1.605 6.776 3.522
H 0.004 6.874 1.643
H 0.762 2.570 4.004
H -0.886 2.690 2.168
H -4.505 3.352 0.287
H -3.212 -0.093 1.497
H -2.629 -2.307 2.406
H 0.187 -5.963 1.854
H -7.795 3.857 1.685
H -5.955 6.307 5.394
H -4.735 8.356 6.103
H 0.783 -2.407 3.492
H 2.046 -0.381 4.184
H 5.239 1.816 6.464
H 4.797 2.371 4.815
H 4.088 3.991 6.607
H 3.051 2.723 7.328
C 11.907 0.451 -0.648
H 12.144 -0.563 -0.303
H 11.370 0.976 0.152
O 4.899 9.922 5.633
O 6.568 3.494 8.217
C 5.124 9.702 6.817
C 5.486 8.374 7.383
C 5.620 7.282 6.514
C 5.959 6.034 7.024
C 6.160 5.862 8.402
C 6.486 4.503 8.912
H 6.674 2.419 10.385
H 5.449 7.425 5.448
H 6.071 5.176 6.365
O 6.393 5.762 2.436
O 5.641 3.634 2.795
O 5.145 8.312 2.454
O 4.812 9.077 0.332
C 6.423 4.547 3.040
C 7.505 4.480 4.061
C 7.740 3.261 4.716
C 8.641 3.205 5.771
C 8.228 5.624 4.436
C 9.139 5.560 5.483
C 9.327 4.358 6.180
C 6.798 10.393 3.103
C 6.105 10.384 1.884
C 5.298 9.212 1.453
C 4.401 7.105 2.152
C 5.298 6.058 1.534
H -7.435 6.817 7.191
H 7.202 2.364 4.414
H 8.804 2.266 6.293
H 8.046 6.570 3.932
H 9.694 6.450 5.768
H 6.762 9.521 3.752
H 3.568 7.324 1.471
H 4.011 6.769 3.121
H 4.720 5.149 1.329
H 5.757 6.418 0.607
O 7.276 7.711 -1.437
O 7.488 9.972 -1.644
O 7.014 -0.568 0.703
O 9.577 -0.722 1.859
O 1.620 9.010 2.207
C 7.460 -1.797 1.327
C 9.723 7.571 1.884
C 8.812 7.636 0.838
C 8.624 8.838 0.141
C 7.740 8.928 -1.053
C 6.033 3.118 -0.791
C 6.367 1.873 -0.280
C 8.968 -1.897 1.273
C 9.822 -0.748 3.194
C 10.459 0.521 3.640
C 3.162 11.260 3.162
C 10.461 1.664 2.825
C 11.153 2.803 3.217
C 11.846 2.812 4.437
H 7.099 -1.732 2.360
H 7.010 -2.666 0.835
H 9.906 6.625 2.389
H 8.258 6.745 0.553
H 4.639 7.408 -1.328
H 5.932 3.963 -0.118
H 6.514 1.749 0.791
H 9.330 -1.908 0.239
H 1.348 7.912 0.070
H 3.188 10.363 3.779
H 9.927 1.648 1.877
H 11.189 3.675 2.569
O -4.227 9.203 -1.560
O -4.054 9.937 0.592
C -5.199 7.115 -0.743
C -3.938 7.845 -1.136
C -4.279 10.157 -0.592
C -4.641 11.485 -1.158
H -4.963 6.101 -0.399
H -5.736 7.640 0.057
H -3.243 7.854 -0.290
O -0.776 10.849 4.018
C -2.318 8.599 3.063
H -2.343 9.497 2.446
H -3.564 7.448 1.737
O -10.092 6.096 0.608
O -7.530 5.941 1.764
O -4.300 11.547 3.771
O -3.967 10.783 5.893
C -8.138 4.767 1.177
C -7.285 5.916 3.098
C -6.647 7.184 3.544
C -5.293 8.347 5.171
C -6.646 8.328 2.729
C -5.954 9.467 3.122
C -5.261 9.475 4.341
C -4.453 10.648 4.772
H -7.776 4.756 0.143
H -7.180 8.312 1.781
H -5.917 10.339 2.473
C 3.944 11.170 -2.358
H 4.181 10.157 -2.013
H 3.407 11.695 -1.558
H 5.580 11.503 0.122
O -0.949 10.151 -1.007
O 1.614 9.997 0.149
C -0.503 8.922 -0.383
C -1.596 12.592 -1.990
C 1.005 8.823 -0.438
C 1.859 9.971 1.483
C 2.496 11.240 1.929
C 2.498 12.383 1.115
H -0.864 8.987 0.649
H -0.953 8.053 -0.875
H -1.449 12.468 -0.920
H 1.367 8.812 -1.472
H 1.964 12.367 0.167
O 10.092 -6.096 -0.608
O 7.530 -5.941 -1.764
O 4.300 -11.547 -3.771
O 3.967 -10.783 -5.893
C 8.138 -4.767 -1.177
C 7.285 -5.916 -3.098
C 6.647 -7.184 -3.544
C 5.293 -8.347 -5.171
C 6.646 -8.328 -2.729
C 5.954 -9.467 -3.122
C 5.261 -9.475 -4.341
C 4.453 -10.648 -4.772
C 9.646 -4.866 -1.232
H 7.776 -4.756 -0.143
H 7.180 -8.312 -1.781
H 5.917 -10.339 -2.473
H 10.008 -4.932 -2.264
H 10.097 -3.998 -0.740
O 0.776 -10.849 -4.018
C 2.318 -8.599 -3.063
H 2.343 -9.497 -2.446
H 3.564 -7.448 -1.737
O 4.227 -9.203 1.560
O 4.054 -9.937 -0.592
C 5.199 -7.115 0.743
C 3.938 -7.845 1.136
C 4.279 -10.157 0.592
C 4.641 -11.485 1.158
H 4.963 -6.101 0.399
H 5.736 -7.640 -0.057
H 3.243 -7.854 0.290
C 0.503 -8.922 0.383
H 0.864 -8.987 -0.649
H 0.953 -8.053 0.875
O -2.415 -3.378 -5.499
O -3.166 -5.506 -5.140
O 1.867 -3.656 -0.178
O 1.656 -5.917 0.029
O 2.129 4.623 -2.318
O -0.433 4.778 -3.474
O 7.523 -4.955 -3.822
O -3.663 -0.828 -5.482
O -3.996 -0.064 -7.603
C -2.384 -4.593 -4.896
C -1.303 -4.660 -3.875
C -1.067 -5.879 -3.220
C -0.167 -5.935 -2.164
C -0.580 -3.516 -3.499
C 0.332 -3.580 -2.452
C 0.520 -4.783 -1.755
C 1.404 -4.872 -0.561
C 3.110 0.937 -0.824
C 2.776 2.183 -1.335
C 0.175 5.952 -2.887
C -0.679 4.804 -4.808
C -1.316 3.535 -5.254
C 5.981 -7.205 -4.777
C -2.670 2.373 -6.881
C -1.317 2.391 -4.440
C -2.009 1.253 -4.832
C -2.703 1.244 -6.052
C -3.510 0.072 -6.483
C -4.407 -2.035 -5.784
C -3.510 -3.082 -6.401
C 1.683 5.853 -2.942
H -1.605 -6.776 -3.522
H -0.004 -6.874 -1.643
H -0.762 -2.570 -4.004
H 0.886 -2.690 -2.168
H 4.505 -3.352 -0.287
H 3.212 0.093 -1.497
H 2.629 2.307 -2.406
H -0.187 5.963 -1.854
H 7.795 -3.857 -1.685
H 5.955 -6.307 -5.394
H 4.735 -8.356 -6.103
H -0.783 2.407 -3.492
H -2.046 0.381 -4.184
H -5.239 -1.816 -6.464
H -4.797 -2.371 -4.815
H -4.088 -3.991 -6.607
H -3.051 -2.723 -7.328
H 2.044 5.788 -3.975
H 2.134 6.721 -2.450
O 2.750 -1.707 -4.051
O 3.502 0.421 -4.410
O -1.532 -1.429 -9.372
O -1.320 0.832 -9.579
O -1.794 -9.708 -7.232
O 0.769 -9.863 -6.076
O -7.188 -0.130 -5.729
O 3.999 -4.257 -4.068
O 4.331 -5.021 -1.947
O -3.401 -3.568 -9.700
C 2.720 -0.492 -4.655
C 1.638 -0.425 -5.676
C 1.403 0.795 -6.331
C 0.502 0.851 -7.386
C 0.915 -1.569 -6.051
C 0.004 -1.505 -7.098
C -0.184 -0.302 -7.795
C -1.068 -0.213 -8.989
C -2.774 -6.022 -8.727
C -2.441 -7.267 -8.215
C 0.160 -11.037 -6.663
C 1.014 -9.888 -4.742
C 1.651 -8.620 -4.296
C -5.646 2.120 -4.773
C 3.006 -7.457 -2.669
C 1.653 -7.476 -5.110
C 2.345 -6.337 -4.718
C 3.038 -6.329 -3.499
C 3.845 -5.156 -3.068
C 4.743 -3.049 -3.767
C 3.846 -2.002 -3.149
C -2.428 -1.480 -10.517
C -3.690 -2.211 -10.125
C -3.349 -4.523 -10.669
C -2.986 -5.851 -10.103
C -2.312 -8.363 -9.084
H 1.941 1.692 -6.029
H 0.339 1.790 -7.907
H 1.098 -2.515 -5.546
H -0.550 -2.395 -7.383
H -4.169 -1.732 -9.263
H -2.876 -5.177 -8.054
H -2.294 -7.392 -7.145
H -7.459 -1.228 -7.865
H -5.620 1.222 -4.157
H -4.400 3.272 -3.448
H 1.119 -7.492 -6.058
H 2.381 -5.465 -5.366
H 5.575 -3.268 -3.086
H 5.132 -2.714 -4.736
H 4.423 -1.093 -2.943
H 3.386 -2.362 -2.222
H -2.665 -0.467 -10.862
H -1.891 -2.005 -11.318
H -4.385 -2.219 -10.970
O 4.072 -6.601 -9.400
O 4.245 -5.867 -7.248
O 2.575 0.561 -9.832
O 2.465 -0.461 -11.869
C 3.099 -8.689 -8.583
C 4.361 -7.959 -8.976
C 4.020 -5.647 -8.432
C 3.657 -4.319 -8.998
C 3.523 -3.227 -8.129
C 3.185 -1.979 -8.639
C 2.983 -1.806 -10.017
C 2.657 -0.447 -10.527
C 3.445 -4.147 -10.374
C 3.112 -2.902 -10.885
C 0.511 0.867 -12.438
C 2.019 0.768 -12.493
H 2.562 -8.164 -7.783
H 5.056 -7.950 -8.130
H 3.695 -3.370 -7.063
H 3.073 -1.120 -7.980
H 3.547 -4.992 -11.047
H 2.965 -2.778 -11.956
H 0.149 0.878 -11.404
H 2.469 1.637 -12.000
O -3.737 1.516 -0.150
O -3.909 0.782 -2.302
O -2.240 -5.646 0.282
C -2.764 3.604 -0.967
C -4.025 2.874 -0.574
C -3.684 0.562 -1.119
C -3.322 -0.766 -0.552
C -3.188 -1.858 -1.421
C -2.849 -3.106 -0.911
C -2.648 -3.278 0.466
C -2.322 -4.637 0.977
H -3.001 4.618 -1.312
H -2.227 3.080 -1.767
H -4.720 2.866 -1.420
H -3.359 -1.715 -2.487
H -2.737 -3.964 -1.571
O 6.729 0.677 -7.114
O 5.977 -1.451 -6.755
O 11.011 0.400 -1.793
O 10.799 -1.861 -1.586
O 5.480 3.228 -7.097
O 5.148 3.992 -9.218
C -7.460 1.797 -1.327
C 6.759 -0.537 -6.510
C 7.841 -0.605 -5.489
C 8.076 -1.824 -4.835
C 8.977 -1.880 -3.779
C 8.564 0.539 -5.114
C 9.475 0.475 -4.067
C 9.663 -0.727 -3.370
C 10.547 -0.817 -2.176
C 7.828 7.590 -6.869
C 6.473 6.428 -8.496
C 7.826 6.447 -6.055
C 7.134 5.308 -6.447
C 6.441 5.299 -7.666
C 5.634 4.127 -8.097
C 4.736 2.020 -7.398
C 5.633 0.973 -8.016
H -7.099 1.732 -2.360
H -7.010 2.666 -0.835
H 7.538 -2.721 -5.137
H 9.140 -2.819 -3.258
H 8.381 1.486 -5.619
H 10.029 1.365 -3.782
H 8.360 6.463 -5.107
H 7.098 4.436 -5.799
H 3.904 2.239 -8.079
H 4.346 1.685 -6.429
H 5.056 0.064 -8.222
H 6.093 1.333 -8.943
O 7.350 -5.653 -8.847
O 1.956 3.926 -7.343
O 5.742 0.487 -11.315
C 10.059 2.487 -7.666
C 9.147 2.551 -8.713
C 6.369 -1.967 -10.341
C 6.703 -3.212 -9.830
C 3.498 6.176 -6.388
H 7.435 -6.817 -7.191
H 10.241 1.541 -7.161
H 8.593 1.661 -8.998
H 4.974 2.323 -10.878
H 6.267 -1.122 -9.669
H 6.849 -3.336 -8.759
H 1.684 2.828 -9.480
H 3.524 5.278 -5.771
H 4.744 7.327 -5.062
H -6.674 -2.419 -10.385
O 5.407 5.572 -1.765
O 5.234 4.838 -3.917
O 6.904 -1.590 -1.333
C 6.379 7.660 -2.582
C 5.118 6.929 -2.189
C 5.459 4.617 -2.733
C 5.822 3.289 -2.167
C 5.956 2.197 -3.036
C 6.294 0.949 -2.526
C 6.496 0.777 -1.148
C 6.822 -0.582 -0.638
H 6.143 8.674 -2.927
H 6.917 7.135 -3.382
H 4.423 6.921 -3.035
H 5.784 2.341 -4.102
H 6.406 0.091 -3.185
O -6.096 7.064 -1.889
O -6.307 4.803 -1.681
O -0.440 5.764 -5.532
C -9.266 6.059 -5.585
C -9.031 4.840 -4.930
C -8.130 4.784 -3.875
C -8.543 7.203 -5.210
C -7.632 7.139 -4.163
C -7.444 5.937 -3.466
C -6.560 5.847 -2.272
C -4.853 11.657 -2.534
C -1.982 3.514 -6.487
H -9.569 3.943 -5.232
H -7.967 3.845 -3.353
H -7.078 8.029 -3.878
H -3.458 7.367 -1.997
H -4.751 10.812 -3.207
H -0.168 6.862 -3.396
H -2.008 4.412 -7.104
H -3.228 2.363 -7.813
O -5.213 9.013 -5.762
O -7.194 0.857 -7.787
O -3.964 6.462 -5.779
O -3.632 5.698 -3.657
C -7.803 -0.318 -8.373
C -6.949 0.831 -6.452
C -6.312 2.100 -6.006
C -4.958 3.262 -4.380
C -6.310 3.243 -6.821
C -5.618 4.382 -6.429
C -4.925 4.391 -5.209
C -4.118 5.563 -4.778
C -3.221 7.670 -5.477
C -4.118 8.717 -4.860
H -6.865 8.204 -7.257
H -7.441 -0.328 -9.407
H -6.844 3.227 -7.769
H -5.582 5.254 -7.077
H -2.388 7.451 -4.796
H -2.831 8.005 -6.446
H -3.540 9.626 -4.654
H -4.577 8.357 -3.932
O -3.891 4.118 -11.111
O -3.718 4.852 -8.959
O -5.760 1.979 -11.439
O -5.972 -0.282 -11.232
C -4.864 2.030 -10.294
C -3.603 2.760 -10.686
C -3.943 5.073 -10.142
H -4.627 1.016 -9.949
H -5.401 2.555 -9.493
H -2.907 2.769 -9.841
H -4.269 7.349 -8.774
H -3.123 2.282 -11.548
O -10.203 5.073 -1.429
C -10.285 6.082 -0.734
O -1.986 9.269 -8.466
O 3.047 11.119 -3.504
O 2.836 8.858 -3.296
C -0.122 10.115 -7.200
C 0.113 8.895 -6.545
C 1.013 8.839 -5.489
C 1.512 11.194 -5.778
C 1.700 9.992 -5.081
C 2.584 9.903 -3.887
H -0.425 7.998 -6.847
H 1.177 7.900 -4.968
H 5.915 6.418 -9.428
O -0.614 5.067 -10.557
O 1.949 4.912 -9.401
O 5.511 9.754 -5.272
C -0.167 3.837 -9.933
C 1.341 3.738 -9.988
C 2.194 4.886 -8.067
C 2.832 6.155 -7.621
C 4.186 7.317 -5.994
C 2.833 7.299 -8.436
C 3.525 8.437 -8.043
C 4.218 8.446 -6.824
C 5.026 9.618 -6.393
C -0.806 5.053 -11.899
H -0.529 3.902 -8.901
H -0.618 2.969 -10.426
H -1.114 7.383 -10.470
H 1.703 3.727 -11.022
H 2.299 7.283 -9.384
O -1.059 9.129 -3.044
C -1.669 11.669 -4.236
C -1.468 11.496 -2.859
C -1.141 10.137 -2.348
H -1.557 10.810 -4.896
"""

# ---------------------------------------------------------------- user interface
load_host(HOST_SITE)
try:
    import ipywidgets as widgets
    from IPython.display import display
    box = widgets.Text(placeholder='e.g.  Cl/C=C/Cl   ClCCCl   FCCF',
                       description='SMILES:', layout=widgets.Layout(width='90%'))
    button = widgets.Button(description='Check', button_style='primary')
    out = widgets.Output()

    def _on_click(_):
        out.clear_output()
        with out:
            print('Working…', flush=True)
        with out:
            out.clear_output(wait=True)
            run(box.value)

    button.on_click(_on_click)
    display(widgets.VBox([box, button, out]))
    print("Ready. Type SMILES in the box above and click 'Check'.")
except Exception:
    print("Ready. To check a molecule, run in a new cell:   run('Cl/C=C/Cl')")


### 🔬 Structural Model

The host geometry is derived from the DFT-optimized trans-1,2-dichloroethylene@C3 structure (PBE-D3, VASP) and remains fixed throughout the analysis.

The binding site consists of two pockets within neighboring C3 rings, connected by a narrow inter-ring region. Each pocket is defined by three aromatic C–H groups oriented toward one end of the guest molecule.

Candidate structures are generated from SMILES using RDKit and MMFF conformational sampling (3 kcal/mol energy window). The two most distant heavy atoms are positioned in opposing pockets, and molecular orientations are systematically sampled to minimize steric overlap.

### 📐 Compatibility Assessment

Geometric compatibility is evaluated using Bondi van der Waals radii and three structural descriptors:

* **Pocket contact** — proximity of each molecular terminus to its binding pocket.
* **Terminal compression** — steric compression experienced by the terminal atoms.
* **Framework overlap** — steric interference with the surrounding host structure.

In addition, each terminal atom must be able to accept C–H···X contacts from its pocket (halogen, O, N or S); carbon termini are classified as incompatible.

The screening criteria are informed by experimentally characterized C3 inclusion complexes and the observed inclusion behavior of guest molecules. These experimental references provide a basis for evaluating geometric compatibility and interpreting the resulting classifications.

### ⚠️ Scope and Limitations

The method assesses molecular compatibility with a specific, experimentally characterized binding geometry. It does not explicitly account for host-framework relaxation, interaction free energies, competing crystalline phases, or inclusion kinetics.

The screening results provide an assessment of structural compatibility, which may help identify promising candidates for inclusion-phase formation.
